# SPU Stormwater EDA — v01

Starter notebook for exploring Seattle Public Utilities' stormwater asset inventory. Two samples are bundled with this notebook, each suited to a different purpose:

| Sample | Location | Purpose |
|---|---|---|
| **`sample/`** | `~/Desktop/spu/sample/` | **Random** sample (~8.7K features, 12 layers). Best for cross-sectional analysis — counts, distributions, ownership splits — where statistical representativeness matters. |
| **`sample_geo/`** | `~/Desktop/spu/sample_geo/` | **Geographic** sample (~18K features, 23 layers) of every asset within a Pioneer Square / SODO bounding box. Best for **mapping** — pipes connect, basins overlap, the network looks like a network. |

Each row in either sample carries the original SPU attributes plus two geometry columns:

| Column | Purpose |
|---|---|
| `geometry_wkt` | Well-Known Text geometry. Universal portable format. |
| `geometry_type` | `Point`, `LineString`, or `Polygon`. |

After running section 5b, every layer also gets `lat` / `lon` columns derived from the geometry.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

SAMPLE     = Path.home() / "Desktop" / "spu" / "sample"      # random sample (cross-sectional)
SAMPLE_GEO = Path.home() / "Desktop" / "spu" / "sample_geo"  # geographic sample (visualization)
GEOMETRY_COLS = ("geometry_wkt", "geometry_type")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## 1. Sample manifest (random sample)

`sample/_index.csv` is the cheat sheet for the random sample — what's in the folder, asset class, geometry type, and how the sample compares to the full layer.

In [ ]:
index = pd.read_csv(SAMPLE / "_index.csv")
index

## 2. Load all (random) sample layers

Stored as a `dict` keyed by the human-readable label. Grab one with `data["Storm Outfalls"]` or iterate.

In [ ]:
data = {row["label"]: pd.read_csv(SAMPLE / row["file"], low_memory=False) for _, row in index.iterrows()}
print(f"Loaded {len(data)} layers, {sum(len(d) for d in data.values()):,} total features")
for label, df in data.items():
    print(f"  {label:<40} {len(df):>6,} rows  x  {len(df.columns):>3} cols")

## 3. Confirm geometry fields are present and populated

Sanity check: each layer should have a non-null `geometry_wkt` for every row, with a consistent `geometry_type`.

In [ ]:
rows = []
for label, df in data.items():
    rows.append({
        "layer": label,
        "rows": len(df),
        "attribute_cols": len(df.columns) - len(GEOMETRY_COLS),
        "geometry_type": df["geometry_type"].iloc[0] if len(df) else "",
        "wkt_populated": df["geometry_wkt"].notna().sum(),
        "sample_wkt": (df["geometry_wkt"].iloc[0][:55] + "...") if len(df) and pd.notna(df["geometry_wkt"].iloc[0]) else "",
    })
pd.DataFrame(rows)

## 4. Cross-sectional exploration (random sample)

For analysis that doesn't need geometry — counts, distributions, correlations — drop the geometry columns for a cleaner view. Below are starter probes; replace with whatever you want to investigate.

### 4a. Storm Outfalls — ownership breakdown

In [ ]:
outfalls = data["Storm Outfalls"]
outfalls_attrs = outfalls.drop(columns=list(GEOMETRY_COLS))
print(f"Storm outfalls — {len(outfalls):,} rows, {len(outfalls_attrs.columns)} attribute columns")
outfalls["OUT_OWNER_NAME"].value_counts(dropna=False).head(10)

### 4b. Mainlines — pipe material distribution

In [ ]:
mains = data["Mainlines Permitted Use (sample)"]
material_col = next((c for c in mains.columns if "MTRL" in c.upper() and "TEXT" in c.upper()), None)
print(f"Using material column: {material_col}")
if material_col:
    mains[material_col].value_counts(dropna=False).head(10)

## 5. Mapping setup — parse WKT to real geometries

1. `shapely.from_wkt` converts WKT strings into geometry objects.
2. Wrap in `geopandas.GeoDataFrame` for spatial joins, projection handling, and built-in plotting.
3. Hand to a mapping library — `folium`, `contextily`, `kepler.gl`, etc.

If the libraries aren't installed yet:

```bash
pip install shapely geopandas folium
```

All sample geometries are in **WGS84 (EPSG:4326)** — lon/lat decimal degrees.

In [ ]:
try:
    from shapely import from_wkt
    import geopandas as gpd
    HAS_GIS = True

    def to_gdf(df: pd.DataFrame, crs: str = "EPSG:4326") -> "gpd.GeoDataFrame":
        """Parse `geometry_wkt` into shapely geometries and return a GeoDataFrame."""
        d = df.copy()
        d["geometry"] = d["geometry_wkt"].apply(from_wkt)
        return gpd.GeoDataFrame(d, geometry="geometry", crs=crs)

    print("GIS libraries available. `to_gdf(df)` helper is ready.")
except ImportError as e:
    HAS_GIS = False
    print(f"Mapping libraries not installed: {e}")
    print("Run:  pip install shapely geopandas folium")

## 5b. Derive uniform `lat` / `lon` on every layer

Adds `lat` and `lon` columns to every random-sample layer:

- **Points**: lat / lon are the point coordinates directly.
- **Lines / Polygons**: lat / lon are the geometry's centroid.

In [ ]:
if HAS_GIS:
    def add_lat_lon(df: pd.DataFrame) -> pd.DataFrame:
        """Add `lat` / `lon` columns derived from `geometry_wkt`.
        Points -> point coords; lines/polygons -> centroid."""
        d = df.copy()
        geoms = d["geometry_wkt"].apply(lambda w: from_wkt(w) if pd.notna(w) else None)
        centroids = geoms.apply(lambda g: g.centroid if g is not None else None)
        d["lat"] = centroids.apply(lambda c: c.y if c is not None else None)
        d["lon"] = centroids.apply(lambda c: c.x if c is not None else None)
        return d

    data = {label: add_lat_lon(df) for label, df in data.items()}

    print(f"{'layer':<40} {'lat/lon%':>9}  {'lat range':<22} {'lon range':<22}")
    print("-" * 100)
    for label, df in data.items():
        ok = df[['lat', 'lon']].notna().all(axis=1).mean() * 100
        print(f"{label:<40} {ok:>8.1f}%  "
              f"({df['lat'].min():.4f}, {df['lat'].max():.4f})  "
              f"({df['lon'].min():.4f}, {df['lon'].max():.4f})")
else:
    print("Run the mapping-setup cell first (shapely is needed).")

## 5c. Load the geographic sample (for mapping)

Random sampling spreads features thin — useful for distributions but produces a sparse, disconnected map. The geographic sample at `~/Desktop/spu/sample_geo/` instead keeps **every** asset within a Pioneer Square / SODO / Elliott Bay bounding box. Same row format (attributes + `geometry_wkt` + `geometry_type`) but features form a connected network.

Loaded into a separate `geo_data` dict so it doesn't conflict with the random-sample `data` dict.

In [ ]:
geo_index = pd.read_csv(SAMPLE_GEO / "_index.csv")
geo_data = {row["label"]: pd.read_csv(SAMPLE_GEO / row["file"], low_memory=False)
            for _, row in geo_index.iterrows()}
print(f"Loaded {len(geo_data)} layers, {sum(len(d) for d in geo_data.values()):,} features")
geo_index[["label", "asset_class", "geometry_type", "n_features_in_geo_sample"]]

## 6. Multi-layer interactive map (geographic sample)

Stacks polygons, lines, and points from `geo_data` on one folium map. The `LayerControl` widget in the corner lets you toggle each layer on/off. Polygon features (basins, footprints) sit on the bottom; line features (mainlines, ditches) in the middle; point features (catch basins, outfalls) on top.

In [ ]:
if HAS_GIS:
    import folium

    def add_layer(m, gdf, name, color, opacity=0.6, point_radius=3):
        if gdf.geometry.iloc[0].geom_type == "Point":
            for _, row in gdf.iterrows():
                folium.CircleMarker(
                    location=[row.geometry.y, row.geometry.x],
                    radius=point_radius, color=color, fill=True, fill_opacity=0.8,
                    popup=name,
                ).add_to(m)
        else:
            folium.GeoJson(
                gdf.to_json(),
                name=name,
                style_function=lambda _: {"color": color, "weight": 2, "fillOpacity": opacity * 0.3},
                tooltip=name,
            ).add_to(m)

    # Pull the layers we care about — order matters: polygons first (bottom), points last (top)
    layer_specs = [
        ("CSO Basins",                  "#7F77DD"),  # purple polygons
        ("GSI Facility Footprints",     "#5DCAA5"),  # teal polygons
        ("Vaults",                      "#1D9E75"),  # dark teal
        ("Pump Stations",               "#534AB7"),
        ("Mainlines (Permitted Use)",   "#185FA5"),  # blue lines
        ("SPU-Owned Non-Mainlines",     "#85B7EB"),  # light blue lines
        ("Ditches & Culverts",          "#0F6E56"),
        ("GSI Swales",                  "#3B6D11"),
        ("Storm Outfalls",              "#993C1D"),  # coral points
        ("Catch Basins",                "#D85A30"),
    ]

    # Center the map on the bbox center
    center = [(47.575 + 47.610) / 2, (-122.350 + -122.315) / 2]
    m = folium.Map(location=center, zoom_start=14, tiles="cartodbpositron")

    for label, color in layer_specs:
        if label not in geo_data:
            continue
        gdf = to_gdf(geo_data[label])
        if len(gdf) == 0:
            continue
        add_layer(m, gdf, label, color)

    folium.LayerControl(collapsed=False).add_to(m)
    m
else:
    print("Run the mapping-setup cell first (shapely / geopandas / folium needed).")

## 7. Where to go from here

- **Cross-sectional probes**: lifecycle status by asset class, install-date distributions, ownership splits between SPU vs private, NPDES coverage rates, dimensions vs material — use the `data` dict (random sample).
- **Spatial probes**: outfalls per drainage basin, GSI density vs catch basin density, distance from each outfall to nearest mainline endpoint — use the `geo_data` dict (geographic sample).
- **Network probes**: join mainlines to their endpoints (via `MNL_UPS_ENDPT_ID` / `MNL_DNS_ENDPT_ID`) to reconstruct topology; trace upstream contributors to a given outfall.
- **Different neighborhood**: edit `BBOX` at the top of `build_geo_sample.py` and rerun to produce a `sample_geo/` for any other area of Seattle.
- **Promotion to the full dataset**: once a probe is shaped well on the sample, swap `SAMPLE` to `~/Desktop/spu/geojson/` (the full `.geojson.gz` files) and use `geopandas.read_file()` instead of `pd.read_csv`.